<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 생성 모델 · 09. Latent Diffusion

## High-Resolution Image Synthesis with Latent Diffusion Models

- **원 논문:** [High-Resolution Image Synthesis with Latent Diffusion Models](https://openaccess.thecvf.com/content/CVPR2022/html/Rombach_High-Resolution_Image_Synthesis_With_Latent_Diffusion_Models_CVPR_2022_paper.html)
- **저자 / 발표:** Robin Rombach, Andreas Blattmann, Dominik Lorenz, Patrick Esser, and Björn Ommer · CVPR (2022)
- **난이도 / 예상 시간:** 고급 · 약 75분
- **선수 지식:** autoencoder, DDPM, latent representation
- **로컬 학습 데이터:** `data/field_curriculum/generative_samples.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 8×8 영상을 8D latent로 압축한 뒤 latent-space noise predictor와 decoder sampling을 구현한다.

**축소하거나 생략한 부분:** 논문은 perceptual autoencoder, convolutional U-Net, cross-attention으로 고해상도 이미지를 생성한다. 여기서는 Eq. (2)의 latent diffusion과 압축 이득에 집중한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1 | perceptual compression | encoder E와 decoder D로 의미 보존 latent space를 만든다. |
| §3.2, Eq. (1)–(2) | latent diffusion objective | pixel x 대신 z=E(x)에서 epsilon prediction을 학습한다. |
| §3.2, Eq. (1)–(2)와 DDPM reverse process | latent reverse sampling과 decoder 복원 | reverse trajectory와 decoded sample의 shape·finite 값을 검증한다. |

## 핵심 재현

        §3.1처럼 먼저 $z=E(x)$와 $\tilde x=D(z)$를 학습합니다. 이어 Eq. (2)의 diffusion
        objective를 64D pixel이 아니라 8D latent에서 실행해 계산 차원을 직접 비교합니다.

In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(391)
np.random.seed(391)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# These teaching batches are intentionally tiny; GPU launch/transfer
# overhead can dominate, so use AI_LAB_DEVICE=cpu for minimum wall time.

DATA_PATH = Path("data/field_curriculum/generative_samples.npz")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
raw = np.load(DATA_PATH)
expected = {"points", "point_labels", "images", "image_labels", "domain_a", "domain_b"}
assert expected.issubset(raw.files), f"NPZ 키 불일치: {sorted(raw.files)}"
points = torch.tensor(raw["points"], dtype=torch.float32)
point_labels = torch.tensor(raw["point_labels"], dtype=torch.long)
images = torch.tensor(raw["images"], dtype=torch.float32)
image_labels = torch.tensor(raw["image_labels"], dtype=torch.long)
domain_a = torch.tensor(raw["domain_a"], dtype=torch.float32)
domain_b = torch.tensor(raw["domain_b"], dtype=torch.float32)
points, point_labels, images, image_labels, domain_a, domain_b = ACCELERATOR.move(
    points, point_labels, images, image_labels, domain_a, domain_b
)


# Seeded random draws stay on CPU so CUDA, MPS, and CPU follow the same
# sampling stream; complete tensors are then transferred to the lab device.
def randn_device(size, *, generator):
    return torch.randn(size, generator=generator).to(DEVICE)


def rand_device(size, *, generator):
    return torch.rand(size, generator=generator).to(DEVICE)


def randint_device(high, size, *, generator):
    return torch.randint(high, size, generator=generator).to(DEVICE)


if float(images.min()) < 0.0:
    images_01 = ((images + 1.0) / 2.0).clamp(0.0, 1.0)
else:
    images_01 = images.clamp(0.0, 1.0)
assert points.shape == (600, 2) and images.shape == (300, 1, 8, 8)
assert domain_a.shape == domain_b.shape == (600, 2)
print(ACCELERATOR.summary())
print("local data:", tuple(points.shape), tuple(images.shape), tuple(domain_a.shape))

## 포트폴리오 구현 설계: 수식 → 책임 → 검증

논문의 핵심 목적을 실행 가능한 객체의 `forward`, `loss`, `update`, `evaluate`로
나눕니다. 아래 식이 클래스 내부에서 생략되지 않고 어떤 tensor 연산이 되는지
메서드별로 추적해 보세요.

$$
\mathcal L_{LDM}=\mathbb E_{z=E(x),t,\epsilon}\lVert\epsilon-\epsilon_\theta(z_t,t)\rVert_2^2
$$

- **기호와 역할:** $E/D$는 perceptual encoder/decoder, $z_t$는 noisy latent, $\epsilon_\theta$는 latent noise predictor입니다.
- **shape 계약:** `image [N, 64] → z [N, 8]; concat(zt,t) [N, 9] → noise [N, 8]`
- **논문 위치:** `§3.1`의 **perceptual compression**
- **코드 Task:** autoencoder와 latent diffusion을 분리 학습하고 8배 차원 축소를 평가합니다.
- **학습·평가 흐름:** AE reconstruction → latent normalize → epsilon update → reverse latent → decode
- **원문 대비 한계:** 논문은 perceptual autoencoder, convolutional U-Net, cross-attention으로 고해상도 이미지를 생성한다. 여기서는 Eq. (2)의 latent diffusion과 압축 이득에 집중한다.
- **구현 이유:** 수식의 tensor 경계와 optimizer 책임을 클래스 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 교육용 편의 함수가 핵심 계산을 대신하지 않도록 loss와 update 순서를
메서드 본문에 드러냅니다. 실습본에서는 같은 공개 API를 유지한 채 TODO를 채우세요.

In [ ]:
class LatentDiffusionLab(nn.Module):
    """Latent Diffusion의 핵심 학습·평가 경로. 입출력 계약: image [N, 64] → z [N, 8];
    concat(zt,t) [N, 9] → noise [N, 8]."""

    def __init__(self, encoder, decoder, noise_predictor, optimizer):
        """구성 요소와 optimizer를 저장한다. 학습은 update에서만 수행한다."""
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.noise_predictor = noise_predictor
        self.optimizer = optimizer
        self.register_buffer("beta_schedule", bz)
        self.register_buffer("alpha_schedule", 1.0 - bz)
        self.register_buffer("alpha_bar", torch.cumprod(1.0 - bz, dim=0))
        self.register_buffer("latent_location", latent_mean)
        self.register_buffer("latent_scale", latent_std)

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {"pixel_dim": 64, "latent_dim": 8, "timesteps": len(self.beta_schedule)}

    def forward(self, clean, timestep, noise):
        """논문 순전파를 계산한다. shape 계약: image [N, 64] → z [N, 8]; concat(zt,t) [N, 9] →
        noise [N, 8]."""
        with torch.no_grad():
            encoded = self.encoder(clean)
            normalized = (encoded - self.latent_location) / self.latent_scale
        signal = self.alpha_bar[timestep, None].sqrt()
        noise_scale = (1.0 - self.alpha_bar[timestep, None]).sqrt()
        noisy_latent = signal * normalized + noise_scale * noise
        predicted_noise = self.predict_noise(noisy_latent, timestep)
        return (noisy_latent, predicted_noise)

    def predict_noise(self, noisy_latent, timestep):
        """논문 알고리즘의 공개 보조 연산을 수행한다."""
        time_feature = timestep[:, None] / (len(self.beta_schedule) - 1)
        return self.noise_predictor(torch.cat([noisy_latent, time_feature], dim=1))

    def loss(self, clean, timestep, noise):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        _, predicted_noise = self.forward(clean, timestep, noise)
        return F.mse_loss(predicted_noise, noise)

    def update(self, clean, timestep, noise):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        objective = self.loss(clean, timestep, noise)
        self.optimizer.zero_grad()
        objective.backward()
        self.optimizer.step()
        return float(objective.detach())

    def reverse_latent_step(self, noisy_latent, step, generator):
        """논문 알고리즘의 공개 보조 연산을 수행한다."""
        timestep = torch.full(
            (len(noisy_latent),), step, dtype=torch.long, device=noisy_latent.device
        )
        predicted_noise = self.predict_noise(noisy_latent, timestep)
        mean = noisy_latent - self.beta_schedule[step] * predicted_noise / torch.sqrt(
            1.0 - self.alpha_bar[step]
        )
        mean = mean / torch.sqrt(self.alpha_schedule[step])
        if step == 0:
            return mean
        innovation = randn_device(noisy_latent.shape, generator=generator)
        return mean + self.beta_schedule[step].sqrt() * innovation

    def sample(self, batch_size, generator):
        """논문 알고리즘의 공개 보조 연산을 수행한다."""
        latent = randn_device(
            (batch_size, self.config()["latent_dim"]), generator=generator
        )
        trajectory = [latent.detach().clone()]
        with torch.no_grad():
            for step in reversed(range(len(self.beta_schedule))):
                latent = self.reverse_latent_step(latent, step, generator)
                trajectory.append(latent.detach().clone())
            decoded = self.decoder(latent * self.latent_scale + self.latent_location)
        return (decoded, latent, torch.stack(trajectory))

    def evaluate(self, clean):
        """no_grad 경로에서 논문과 연결된 정량 지표 dict를 반환한다."""
        with torch.no_grad():
            reconstruction = self.decoder(self.encoder(clean))
            reconstruction_mse = F.mse_loss(reconstruction, clean)
            sample, sampled_latent, trajectory = self.sample(
                min(8, len(clean)), torch.Generator().manual_seed(4091)
            )
        return {
            "reconstruction_mse": float(reconstruction_mse),
            "compression_ratio": self.config()["pixel_dim"]
            / self.config()["latent_dim"],
            "sample_shape": tuple(sample.shape),
            "sampled_latent_shape": tuple(sampled_latent.shape),
            "trajectory_steps": len(trajectory),
            "sample_pixel_mean": float(sample.mean()),
        }

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§3.1` — **perceptual compression**
- **구현 이유:** encoder E와 decoder D로 의미 보존 latent space를 만든다.
- **읽을 코드:** 아래 `compression` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
E = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 8)).to(DEVICE)
Dec = nn.Sequential(nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, 64), nn.Sigmoid()).to(
    DEVICE
)
x_data = images_01.flatten(1)
ae_opt = torch.optim.Adam(list(E.parameters()) + list(Dec.parameters()), lr=0.01)
ae_history = []
with torch.no_grad():
    initial_ae = float(F.mse_loss(Dec(E(x_data)), x_data))
for _ in range(140):
    rec = Dec(E(x_data))
    loss = F.mse_loss(rec, x_data)
    ae_opt.zero_grad()
    loss.backward()
    ae_opt.step()
    ae_history.append(float(loss.detach()))
assert ae_history[-1] < initial_ae and E(x_data).shape == (300, 8)

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§3.2, Eq. (1)–(2)` — **latent diffusion objective**
- **구현 이유:** pixel x 대신 z=E(x)에서 epsilon prediction을 학습한다.
- **읽을 코드:** 아래 `latent-diffusion` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
with torch.no_grad():
    latent = E(x_data)
    latent_mean = latent.mean(0)
    latent_std = latent.std(0).clamp_min(0.1)
    latent = (latent - latent_mean) / latent_std
Tz = 16
bz = torch.linspace(1e-4, 0.10, Tz, device=DEVICE)
az = 1 - bz
abz = torch.cumprod(az, 0)
eps_model = nn.Sequential(nn.Linear(9, 48), nn.SiLU(), nn.Linear(48, 8)).to(DEVICE)
opt = torch.optim.Adam(eps_model.parameters(), lr=0.005)
rng = torch.Generator().manual_seed(392)
diff_history = []

portfolio_model = LatentDiffusionLab(E, Dec, eps_model, opt)
initial_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in eps_model.parameters()]
)
for _ in range(180):
    timestep = randint_device(Tz, (len(latent),), generator=rng)
    noise = randn_device(latent.shape, generator=rng)
    diff_history.append(portfolio_model.update(x_data, timestep, noise))
final_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in eps_model.parameters()]
)
parameter_delta = float((final_parameters - initial_parameters).abs().sum())
assert parameter_delta > 0.0 and np.isfinite(diff_history).all()
assert np.mean(diff_history[-20:]) < np.mean(diff_history[:20])

### 단계 4. 평가·시각화

        - **논문의 어느 부분인가:** `§3.2, Eq. (1)–(2)와 DDPM reverse process` — **latent reverse sampling과 decoder 복원**
        - **구현 이유:** reverse trajectory와 decoded sample의 shape·finite 값을 검증한다.
        - **읽을 코드:** 아래 `evaluation` 셀에서 중간 tensor의 shape, gradient가 흐르는
          경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

        실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
        주석으로 적은 뒤 실습하세요.
        - **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

- **평가 범위:** 이 값은 작은 학습 batch의 메커니즘 진단이며, held-out 생성 품질이나 원 논문의 benchmark 성능으로 해석하지 않습니다.

In [ ]:
portfolio_metrics = portfolio_model.evaluate(x_data)
assert portfolio_metrics

rng2 = torch.Generator().manual_seed(393)
z = randn_device((8, 8), generator=rng2)
with torch.no_grad():
    for step in reversed(range(Tz)):
        t = torch.full((len(z),), step, dtype=torch.long, device=DEVICE)
        pred = eps_model(torch.cat([z, t[:, None] / (Tz - 1)], 1))
        mean = (z - bz[step] / torch.sqrt(1 - abz[step]) * pred) / torch.sqrt(az[step])
        z = (
            mean
            if step == 0
            else mean + bz[step].sqrt() * randn_device(z.shape, generator=rng2)
        )
    decoded = Dec(z * latent_std + latent_mean)
ratio = 64 / 8
print(
    f"diffusion dimension: 64 -> 8 ({ratio:.0f}x smaller), AE MSE={ae_history[-1]:.4f}"
)
assert ratio == 8 and decoded.shape == (8, 64) and torch.isfinite(decoded).all()
fig, axes = plt.subplots(2, 5, figsize=(9, 4))
axes[0, 0].plot(ae_history)
axes[0, 0].set_title("AE loss")
axes[1, 0].plot(diff_history)
axes[1, 0].set_title("latent DM loss")
for ax, img in zip(axes[:, 1:].ravel(), decoded.detach().cpu()):
    ax.imshow(img.reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
fig.tight_layout()
plt.show()

encoder가 고주파와 세부 구조를 제거하면 diffusion이 latent 분포를 완벽히 학습해도 decoder가 그 정보를 복원할 수 없습니다. §4.1의 downsampling-factor trade-off입니다.

### 단계 5. 통합 클래스가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§3.1`의 **perceptual compression**
- **핵심 식:**

  $$
  \mathcal L_{LDM}=\mathbb E_{z=E(x),t,\epsilon}\lVert\epsilon-\epsilon_\theta(z_t,t)\rVert_2^2
  $$

- **입출력 shape:** `image [N, 64] → z [N, 8]; concat(zt,t) [N, 9] → noise [N, 8]`
- **구현 이유:** 앞 셀은 수식을 작은 연산으로 분해해 확인하고,
  이 셀은 같은 구성 요소를 `LatentDiffusionLab`에 주입해
  최종 학습·평가 경로를 하나로 묶습니다. encoder E와 decoder D로 의미 보존 latent space를 만든다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 논문 대응 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_model.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(diff_history) > 0
assert np.isfinite(np.asarray(diff_history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_model).__name__,
        "steps": len(diff_history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 perceptual autoencoder, convolutional U-Net, cross-attention으로 고해상도 이미지를 생성한다. 여기서는 Eq. (2)의 latent diffusion과 압축 이득에 집중한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.